# 🎬 VSL Forge Service — MECProAI
## Google Colab + ngrok

Este notebook roda o VSL Service e expõe via ngrok para o MECProAI usar.

### ⚠️ ANTES DE RODAR:
1. Vá em **Runtime → Change runtime type → T4 GPU**
2. Preencha suas API keys na célula de configuração
3. Execute as células em ordem

### ⏱️ Tempo estimado por vídeo:
- Colab T4 GPU: ~3-8 minutos por vídeo
- Colab CPU: ~15-30 minutos por vídeo

## 📦 CÉLULA 1 — Instalar dependências

In [ ]:
# Instalar dependências do sistema
!apt-get update -qq
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ FFmpeg instalado')

# Verificar FFmpeg
!ffmpeg -version | head -1

# Instalar dependências Python
!pip install -q fastapi uvicorn python-multipart aiofiles pyngrok nest-asyncio
!pip install -q requests Pillow tqdm elevenlabs huggingface_hub
print('✅ Dependências Python instaladas')

## 🔑 CÉLULA 2 — Configurar API Keys

In [ ]:
import os

# ══════════════════════════════════════════════
# PREENCHA SUAS KEYS AQUI
# ══════════════════════════════════════════════

ELEVENLABS_API_KEY = "sk_COLE_SUA_KEY_AQUI"     # elevenlabs.io
HUGGINGFACE_API_KEY = "hf_COLE_SUA_KEY_AQUI"    # huggingface.co/settings/tokens
NGROK_AUTH_TOKEN = "COLE_SEU_TOKEN_AQUI"         # dashboard.ngrok.com
VSL_SECRET_KEY = "mecpro-vsl-secret"             # mesmo valor do Render

# ══════════════════════════════════════════════

os.environ['ELEVENLABS_API_KEY'] = ELEVENLABS_API_KEY
os.environ['HUGGINGFACE_API_KEY'] = HUGGINGFACE_API_KEY
os.environ['VSL_SECRET_KEY'] = VSL_SECRET_KEY
os.environ['OUTPUT_DIR'] = '/content/vsl_outputs'

os.makedirs('/content/vsl_outputs', exist_ok=True)

# Verificar
print('✅ ElevenLabs:', 'OK' if ELEVENLABS_API_KEY != 'sk_COLE_SUA_KEY_AQUI' else '❌ NÃO CONFIGURADO')
print('✅ HuggingFace:', 'OK' if HUGGINGFACE_API_KEY != 'hf_COLE_SUA_KEY_AQUI' else '❌ NÃO CONFIGURADO')
print('✅ ngrok:', 'OK' if NGROK_AUTH_TOKEN != 'COLE_SEU_TOKEN_AQUI' else '❌ NÃO CONFIGURADO')

## 📥 CÉLULA 3 — Fazer upload dos scripts VSL

In [ ]:
from google.colab import files
import shutil

print('📤 Faça upload dos arquivos:')
print('  - vsl_forge_v2.py')
print('  - vsl_service.py')
print('')

uploaded = files.upload()

for filename in uploaded:
    dest = f'/content/{filename}'
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f'✅ {filename} → {dest}')

# Verificar
import os
forge_ok   = os.path.exists('/content/vsl_forge_v2.py')
service_ok = os.path.exists('/content/vsl_service.py')
print(f'\nvsl_forge_v2.py:  {"✅" if forge_ok else "❌ faltando"}')
print(f'vsl_service.py:   {"✅" if service_ok else "❌ faltando"}')

## 🧪 CÉLULA 4 — Testar geração de vídeo (opcional)

In [ ]:
import json, subprocess, sys

# Projeto de teste rápido
test_project = {
  'title': 'Teste MECProAI',
  'output_name': 'teste_mecpro',
  'tts_engine': 'elevenlabs',
  'voice': 'hpp4J3VqNfWAUOO0d1Us',
  'add_subtitles': True,
  'color_grade': 'warm_film',
  'vignette': True,
  'crossfade_duration': 0.8,
  'normalize_audio': True,
  'master_music_gain_db': -20,
  'scenes': [
    {
      'id': 'hook',
      'narration': 'Descubra como dobrar seus resultados com inteligência artificial.',
      'prompt': 'Professional Brazilian entrepreneur smiling at laptop, modern office, cinematic lighting',
      'motion': 'push_in',
      'xfade': 'dissolve',
      'xfade_duration': 0.8,
      'cinematic_bars': False,
      'grain': 0.06,
      'color_grade': 'warm_film',
      'audio_fade_in': 0.05,
      'audio_fade_out': 0.3,
    }
  ]
}

with open('/content/test_project.json', 'w') as f:
    json.dump(test_project, f, indent=2)

print('▶️ Rodando teste em modo draft (rápido)...')
result = subprocess.run(
    [sys.executable, '/content/vsl_forge_v2.py',
     '--config', '/content/test_project.json',
     '--draft'],
    capture_output=False,
    text=True
)
print('\nCódigo de saída:', result.returncode)
if result.returncode == 0:
    print('✅ Teste passou! VSL Forge funcionando.')
else:
    print('❌ Teste falhou. Verifique os logs acima.')

## 🚀 CÉLULA 5 — Iniciar VSL Service + ngrok

In [ ]:
import nest_asyncio
import uvicorn
import threading
import time
from pyngrok import ngrok, conf

nest_asyncio.apply()

# Configurar ngrok
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Importar o app FastAPI
import sys
sys.path.insert(0, '/content')

# Ajusta o caminho do VSL script no service
import importlib.util
spec = importlib.util.spec_from_file_location('vsl_service', '/content/vsl_service.py')
vsl_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vsl_module)
app = vsl_module.app

# Rodar FastAPI em thread separada
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# Abrir túnel ngrok
tunnel = ngrok.connect(8000, 'http')
public_url = tunnel.public_url

print('=' * 60)
print('🚀 VSL Service rodando!')
print('=' * 60)
print(f'URL pública: {public_url}')
print(f'Health check: {public_url}/health')
print('')
print('📋 COLE ESTAS VARIÁVEIS NO RENDER:')
print(f'VSL_SERVICE_URL={public_url}')
print(f'VSL_SECRET_KEY={VSL_SECRET_KEY}')
print('=' * 60)
print('')
print('⚠️  IMPORTANTE: A URL muda toda vez que reiniciar este notebook!')
print('    Atualize o VSL_SERVICE_URL no Render sempre que reiniciar.')

## 🔍 CÉLULA 6 — Verificar se está funcionando

In [ ]:
import requests

headers = {'Authorization': f'Bearer {VSL_SECRET_KEY}'}

# Health check
r = requests.get(f'{public_url}/health', headers=headers)
data = r.json()

print('🔍 Status do serviço:')
print(f'  FFmpeg:      {"✅" if data.get("ffmpeg") else "❌"}')
print(f'  HuggingFace: {"✅" if data.get("huggingface") else "❌"}')
print(f'  ElevenLabs:  {"✅" if data.get("elevenlabs") else "❌"}')
print(f'  Jobs ativos: {data.get("jobs_active", 0)}')
print()

# Listar formatos
r2 = requests.get(f'{public_url}/formats', headers=headers)
formats = r2.json()
print('📋 Formatos disponíveis:')
for k, v in formats.items():
    print(f'  {k}: {v["label"]} ({v["resolution"]})')

## 🎬 CÉLULA 7 — Testar geração via API (opcional)

In [ ]:
import requests, time

headers = {'Authorization': f'Bearer {VSL_SECRET_KEY}', 'Content-Type': 'application/json'}

# Payload de teste
payload = {
    'title': 'Teste via API',
    'format': 'reels_9_16',
    'project_name': 'teste_api',
    'draft': True,  # modo rápido
    'scenes': [
        {
            'id': 'hook',
            'narration': 'MECProAI transforma suas campanhas com inteligência artificial.',
            'prompt': 'Brazilian professional woman smiling at smartphone, modern city background, cinematic',
            'motion': 'push_in',
            'xfade': 'dissolve',
            'xfade_duration': 0.8,
            'cinematic_bars': False,
            'grain': 0.06,
            'color_grade': 'warm_film',
            'audio_fade_in': 0.05,
            'audio_fade_out': 0.3,
        }
    ]
}

# Iniciar geração
r = requests.post(f'{public_url}/generate', json=payload, headers=headers)
job = r.json()
job_id = job['job_id']
print(f'✅ Job iniciado: {job_id}')
print(f'Estimativa: {job["estimated_min"]} minutos')

# Polling de status
print('\nAguardando...')
while True:
    time.sleep(10)
    r2 = requests.get(f'{public_url}/status/{job_id}', headers=headers)
    status = r2.json()
    print(f'  [{status["progress"]}%] {status["message"]}')
    if status['status'] in ['done', 'error']:
        break

if status['status'] == 'done':
    print(f'\n🎉 Vídeo pronto!')
    print(f'URL: {public_url}{status["video_url"]}')
    print(f'Duração: {status["duration_s"]}s | Tamanho: {status["size_mb"]}MB')
else:
    print(f'\n❌ Erro: {status["error"]}')

## 📌 CÉLULA 8 — Manter vivo (anti-timeout)
Execute esta célula para o Colab não desconectar enquanto processa vídeos.

In [ ]:
import time, threading

print('🔄 Anti-timeout ativo. O Colab não vai desconectar.')
print('   Para parar: Interrompa a célula (■)')
print(f'\n🌐 URL do serviço: {public_url}')
print(f'📋 VSL_SERVICE_URL={public_url}')

contador = 0
while True:
    time.sleep(60)
    contador += 1
    # Verifica jobs ativos
    try:
        import requests
        r = requests.get(f'{public_url}/health',
                        headers={'Authorization': f'Bearer {VSL_SECRET_KEY}'},
                        timeout=5)
        jobs = r.json().get('jobs_active', 0)
        print(f'  ⏱️ {contador}min — Serviço OK | Jobs ativos: {jobs}')
    except:
        print(f'  ⚠️ {contador}min — Serviço sem resposta')